# Full LISA Response vs. Long-Wavelength Approximation (LWA)

This notebook justifies using the **full BBHx LISA TDI response** rather than the
long-wavelength approximation (LWA) for the inspiral-only MBHB pipeline.

## Background

The LWA treats the LISA arm length as negligible compared to the GW wavelength — valid
when `f ≪ f* = c/(2πL) ≈ 19 mHz` (for LISA arm `L = 2.5 Gm`).

For our MBHB mass range 10⁵–10⁶ M☉:
- **10⁶ M☉** → f_ISCO ≈ 4.4 mHz  (mostly below f*, LWA may be adequate)
- **10⁵ M☉** → f_ISCO ≈ 44 mHz   (well above f*, full response needed)

We compute the **mismatch** between full-response waveforms (BBHx) and LWA waveforms
(LISABeta with `responseapprox='LWA'`) to quantify the approximation error.

**Run this notebook in the cluster environment** after bootstrapping with
`misc_scripts/lisa_cluster_bootstrap.sh` or the equivalent SLURM setup.

In [ ]:
from pathlib import Path
import sys
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

repo_root = Path.cwd()
if not (repo_root / 'misc_scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from misc_scripts.compare_lisa_bbhx_lisabeta import (
    environment_report,
    _require_waveform_stack,
    DEFAULT_LISA_SETTINGS,
    PYCONSTANTS_YRSID_SI,
)

print(environment_report())

In [ ]:
_require_waveform_stack()

from dingo.gw.waveform_generator.waveform_generator import (
    BBHxWaveformGenerator,
    LISAWaveformGenerator,
)
from dingo.gw.domains import build_domain
from dingo.gw.transforms.detector_transforms import ProjectOntoSpaceDetectors

print('Imports OK')

## Setup: domain, parameters, waveform generators

In [ ]:
# Uniform frequency domain covering the inspiral band for our mass range.
# f_ISCO(10^6 Msun) ≈ 4.4 mHz,  f_ISCO(10^5 Msun) ≈ 44 mHz
DOMAIN_SETTINGS = dict(
    type='UniformFrequencyDomain',
    f_min=1e-4,
    f_max=1e-1,
    delta_f=5e-6,
)
domain = build_domain(DOMAIN_SETTINGS)
freqs = np.array(domain.sample_frequencies)
print(f'Domain: {len(freqs)} bins, {freqs[0]:.2e}–{freqs[-1]:.2e} Hz')

# LISA characteristic frequency (arm-length scale)
LISA_ARM_M   = 2.5e9      # metres
C_SI         = 3e8        # m/s
F_STAR       = C_SI / (2 * math.pi * LISA_ARM_M)
print(f'LISA characteristic frequency f* ≈ {F_STAR*1e3:.1f} mHz')

In [ ]:
# Shared (sky + orientation) parameters — same for both mass cases.
EXTRINSIC = dict(
    inc   = math.pi / 4,
    phi   = 0.0,
    lam   = 0.5,      # ecliptic longitude (rad)
    beta  = 0.3,      # ecliptic latitude  (rad)
    psi   = 1.2,
    dist  = 5000.0,   # Mpc
    geocent_time = 0.25 * PYCONSTANTS_YRSID_SI,  # t_ref (s), merger at 0.75 yr
)

# Two test cases: heavy (mostly in LWA regime) and light (above f*).
CASES = {
    '1e6_Msun': dict(Mchirp=7.0e5, q=0.9, chi1=0.3, chi2=0.3),
    '1e5_Msun': dict(Mchirp=7.0e4, q=0.9, chi1=0.3, chi2=0.3),
}
for name, p in CASES.items():
    Mchirp = p['Mchirp']
    q      = p['q']
    eta    = q / (1 + q)**2
    M_tot  = Mchirp / eta**0.6
    f_isco = 4400.0 / M_tot
    print(f'{name}: M_tot ≈ {M_tot:.2e} Msun,  f_ISCO ≈ {f_isco*1e3:.1f} mHz')

In [ ]:
# BBHx generator — full response
bbhx_gen = BBHxWaveformGenerator(
    approximant='PhenomHM',
    domain=DOMAIN_SETTINGS,
    f_ref=1e-3,
    use_gpu=False,
    direct_response=False,
    bbhx_t_obs_start_years=0.0,
    bbhx_t_obs_end_years=0.25,
    default_t_ref_years=0.75,
    bbhx_length=2048,
    isco_cutoff=True,
)

# LISABeta generator — intrinsic amplitude/phase
lisabeta_gen = LISAWaveformGenerator(
    approximant='IMRPhenomHM',
    domain=DOMAIN_SETTINGS,
    f_ref=1e-3,
)

# LISABeta response transform — FULL response
lisa_settings_full = dict(DEFAULT_LISA_SETTINGS)
lisa_settings_full['responseapprox'] = 'full'

# LISABeta response transform — LWA
lisa_settings_lwa = dict(DEFAULT_LISA_SETTINGS)
lisa_settings_lwa['responseapprox'] = 'LWA'

ref_time = 0.0
channels = ['chan1', 'chan2', 'chan3']
proj_full = ProjectOntoSpaceDetectors('TDIAET', domain, ref_time, channels, lisa_settings_full)
proj_lwa  = ProjectOntoSpaceDetectors('TDIAET', domain, ref_time, channels, lisa_settings_lwa)

print('Generators and projections initialised')

## LISA PSD helper (ESA Proposal noise budget)

Used to weight the overlap integral.

In [ ]:
def lisa_psd_A(f, L=2.5e9, f_star=None):
    """Approximate TDI-A channel PSD (ESA Proposal noise budget).
    
    Parameters
    ----------
    f : array_like, Hz
    L : float, LISA arm length in metres
    
    Returns
    -------
    S_A : ndarray, 1/Hz
    """
    f = np.asarray(f, dtype=float)
    if f_star is None:
        f_star = C_SI / (2 * math.pi * L)

    # OMS (position) noise
    S_oms = (1.5e-11)**2 * (1 + (2e-3 / f)**4)  # m²/Hz

    # Test-mass (acceleration) noise — converted to displacement
    S_acc = (3e-15)**2 * (1 + (4e-4 / f)**2) * (1 + (f / 8e-3)**4)
    S_acc /= (2 * math.pi * f)**4  # m²/Hz

    # Single-link optical metrology + TM noise
    S_link = S_oms / L**2 + 2 * S_acc / L**2  # fractional frequency

    # TDI-A combination (3-arm geometry)
    x = 2 * math.pi * f * L / C_SI
    S_A = 8 * np.sin(x)**2 * (2 * (1 + np.cos(x)**2) * S_link)
    return S_A


def overlap(h1, h2, psd, df):
    """Frequency-domain overlap <h1|h2> / sqrt(<h1|h1><h2|h2>)."""
    inner = lambda a, b: 4 * df * np.real(np.sum(np.conj(a) * b / psd))
    return inner(h1, h2) / math.sqrt(inner(h1, h1) * inner(h2, h2))


df   = domain.delta_f
psd  = lisa_psd_A(freqs)
psd  = np.where(freqs > 0, psd, np.inf)  # avoid divide-by-zero at f=0
print(f'PSD shape: {psd.shape}')

## Generate waveforms and compute mismatch

In [ ]:
def build_params(intrinsic, extrinsic):
    """Merge intrinsic + extrinsic into a single parameter dict."""
    p = {}
    p.update(intrinsic)
    p.update(extrinsic)
    return p


def get_bbhx_strains(gen, intrinsic, extrinsic):
    """Return dict of chan1/chan2/chan3 complex strains from BBHx."""
    params = build_params(intrinsic, extrinsic)
    wf = gen.generate_amp_phase(params)
    h = wf['waveform']  # shape (3, n_freqs)
    return {'chan1': h[0], 'chan2': h[1], 'chan3': h[2]}


def get_lisabeta_strains(gen, proj, intrinsic, extrinsic):
    """Return dict of chan1/chan2/chan3 complex strains from LISABeta."""
    params = build_params(intrinsic, extrinsic)
    wf = gen.generate_amp_phase_m(params)
    sample = {'waveform': wf, 'parameters': params, 'extrinsic_parameters': extrinsic}
    out = proj(sample)
    return out['waveform']  # dict: chan1, chan2, chan3


results = {}

for case_name, intrinsic in CASES.items():
    print(f'\n=== {case_name} ===')

    # Full response from BBHx
    h_bbhx = get_bbhx_strains(bbhx_gen, intrinsic, EXTRINSIC)

    # Full response from LISABeta
    h_full = get_lisabeta_strains(lisabeta_gen, proj_full, intrinsic, EXTRINSIC)

    # LWA from LISABeta
    h_lwa  = get_lisabeta_strains(lisabeta_gen, proj_lwa,  intrinsic, EXTRINSIC)

    case_results = {}
    for ch in ['chan1', 'chan2']:
        ov_full_lwa  = overlap(h_full[ch], h_lwa[ch],  psd, df)
        ov_bbhx_full = overlap(h_bbhx[ch], h_full[ch], psd, df)
        mm_full_lwa  = 1.0 - ov_full_lwa
        mm_bbhx_full = 1.0 - ov_bbhx_full
        print(f'  {ch}  mismatch(full vs LWA) = {mm_full_lwa:.4e}   '
              f'mismatch(BBHx vs LISAbeta-full) = {mm_bbhx_full:.4e}')
        case_results[ch] = dict(
            h_bbhx=h_bbhx[ch],
            h_full=h_full[ch],
            h_lwa=h_lwa[ch],
            mm_full_lwa=mm_full_lwa,
            mm_bbhx_full=mm_bbhx_full,
        )
    results[case_name] = case_results

## Amplitude ratio: full response vs. LWA

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

channel_labels = {'chan1': 'TDI-A', 'chan2': 'TDI-E'}

for row, (case_name, case_res) in enumerate(results.items()):
    Mchirp = CASES[case_name]['Mchirp']
    q      = CASES[case_name]['q']
    eta    = q / (1 + q)**2
    M_tot  = Mchirp / eta**0.6
    f_isco = 4400.0 / M_tot

    for col, ch in enumerate(['chan1', 'chan2']):
        ax  = axes[row, col]
        h_f = case_res[ch]['h_full']
        h_l = case_res[ch]['h_lwa']

        mask = np.abs(h_f) > 0
        ratio = np.where(mask, np.abs(h_f) / np.maximum(np.abs(h_l), 1e-300), np.nan)

        ax.semilogx(freqs * 1e3, ratio, lw=0.8, color='steelblue')
        ax.axvline(F_STAR * 1e3, color='red',    ls='--', lw=1.2, label=f'f* = {F_STAR*1e3:.0f} mHz')
        ax.axvline(f_isco * 1e3, color='orange', ls=':',  lw=1.2, label=f'f_ISCO = {f_isco*1e3:.1f} mHz')
        ax.axhline(1.0, color='k', ls=':', lw=0.7)
        ax.set_ylim(0.5, 1.5)
        ax.set_ylabel('|h_full| / |h_LWA|')
        ax.set_title(f'{case_name}, {channel_labels[ch]}\n'
                     f'mismatch = {case_res[ch]["mm_full_lwa"]:.2e}')
        if row == 0:
            ax.legend(fontsize=8)

axes[-1, 0].set_xlabel('Frequency [mHz]')
axes[-1, 1].set_xlabel('Frequency [mHz]')
fig.suptitle('Full LISA response vs. Long-Wavelength Approximation', fontsize=13)
fig.tight_layout()
plt.savefig('full_vs_lwa_amplitude_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved full_vs_lwa_amplitude_ratio.png')

## Mismatch vs. mass — sky-position sweep

In [ ]:
import itertools

rng = np.random.default_rng(42)
N_SKY = 8

sky_positions = [
    dict(
        lam  = rng.uniform(0, 2 * math.pi),
        beta = np.arcsin(rng.uniform(-1, 1)),
        psi  = rng.uniform(0, math.pi),
    )
    for _ in range(N_SKY)
]

# Sample log-uniformly in total mass within the prior
log_M_min, log_M_max = np.log10(1e5), np.log10(1e6)
test_masses = np.logspace(log_M_min, log_M_max, 8)  # 8 mass values

mm_grid = np.zeros((len(test_masses), N_SKY))

for i, M_tot in enumerate(test_masses):
    # Approximate intrinsic params from total mass (fixed q, chi)
    q = 0.9
    eta = q / (1 + q)**2
    Mchirp = M_tot * eta**0.6
    intr = dict(Mchirp=Mchirp, q=q, chi1=0.3, chi2=0.3)

    for j, sky in enumerate(sky_positions):
        ext = dict(EXTRINSIC)
        ext.update(sky)
        try:
            h_f = get_lisabeta_strains(lisabeta_gen, proj_full, intr, ext)
            h_l = get_lisabeta_strains(lisabeta_gen, proj_lwa,  intr, ext)
            mm_grid[i, j] = 1.0 - overlap(h_f['chan1'], h_l['chan1'], psd, df)
        except Exception as e:
            print(f'  M={M_tot:.2e}, sky={j}: {e}')
            mm_grid[i, j] = np.nan

    print(f'M_tot = {M_tot:.2e} Msun:  mismatch median = {np.nanmedian(mm_grid[i]):.3e}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.fill_between(
    test_masses,
    np.nanpercentile(mm_grid, 10, axis=1),
    np.nanpercentile(mm_grid, 90, axis=1),
    alpha=0.3, color='steelblue', label='10–90th percentile'
)
ax.semilogx(test_masses, np.nanmedian(mm_grid, axis=1),
            'o-', color='steelblue', label='Median mismatch')
ax.axhline(0.01, color='red', ls='--', lw=1.2, label='1% threshold')
ax.set_xlabel('Total mass [M☉]')
ax.set_ylabel('Mismatch  (1 − overlap)')
ax.set_title('Full response vs. LWA: mismatch vs. total mass (TDI-A)')
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.5)
fig.tight_layout()
plt.savefig('full_vs_lwa_mismatch_vs_mass.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved full_vs_lwa_mismatch_vs_mass.png')

## Summary

- If the mismatch is negligible (< 0.1%) for the full mass range, the LWA is sufficient.
- If the mismatch exceeds ~1% for lower-mass sources (M < ~3×10⁵ M☉), the full response
  must be used — which is what BBHx provides.
- The plot above shows how the approximation error grows with total mass (lower mass =
  higher f_ISCO = more signal above f*).